# Phishing URL Detection

The x is done in the following steps:
1. Loading data
2. Familiarizing with data and EDA
3. Visualizing the data
4. Splitting the data
5. Model Building and Training
    
    5.1 Logistic Regression

    5.2 Decision Tree

    5.3 Random Forest

    5.4 Support Vector Machine

    5.5 XGBoost
    
6. Comparison of Models
7. Storing the Best Model
8. Feature Importance Analysis
9. Conclusion

In [ ]:
# Importing required libraries

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline
import seaborn as sns
from sklearn import metrics
import warnings

pd.set_option("display.max_columns", None)

## 1. Loading data:

The dataset taken from the UCI Machine Learning Repository on Phishing Websites (available [here](https://archive.ics.uci.edu/dataset/327/phishing+websites)) is a collection of 11,000+ websites, where each of the samples has 30 website parameters and a class label identifying it as a phishing website or not (1 or -1).

In [ ]:
# Loading data into dataframe
df = pd.read_csv("phishing.csv")
df.head()

## 2. Familiarizing with data and EDA:

Looking into a few dataframe methods to understand the data and its features, including Exploratory Data Analysis (EDA).

In [ ]:
df.shape

In [ ]:
df.columns

In [ ]:
df.info()

In [ ]:
# Dropping Index column
df = df.drop(['Index'], axis=1)

In [ ]:
# Statistical Summary of Dataset
df.describe().T

In [ ]:
# Converting target labels from -1 for phishing to 1, and from 1 to 0 for legitimate urls.

df['class'] = df['class'].map({
    -1: 1,  # Phishing
     1: 0   # Legitimate
})

In [ ]:
# Check for number of legitimate and phishing urls.

print("Target values after conversion:")
print(df['class'].value_counts())

print("\nTarget labels:")
print(df['class'].map({
    1: "Phishing",
    0: "Legitimate"
}).value_counts())

### EDA Summary

1. There are 11,054 instances and 31 features.
2. Each feature is in `int` datatype, so there is no need to use LabelEncoder.
3. Missing values: `df.info()` shows no missing values in any column.
4. Target column: `class` is what we'll predict.
4. There is no outlier present in dataset.

## 3. Visualizing the data:

Looking into a few plots and graphs to find how the data is distributed and how the features are related to each other.

In [ ]:
# Target class distribution

plt.figure(figsize=(7, 5))

ax = sns.countplot(data=df, x='class', hue='class', palette='Set2', legend=False)

plt.title('Distribution of Phishing and Legitimate Websites')
plt.xlabel('Website Class')
plt.ylabel('Number of Websites')
plt.xticks([0, 1], ['Legitimate (0)', 'Phishing (1)'])

total = len(df)

for bar in ax.patches:
    count = int(bar.get_height())
    percentage = count / total * 100
    ax.annotate(f'{count}\n({percentage:.2f}%)', (bar.get_x() + bar.get_width() / 2, count), ha='center', va='bottom')

plt.ylim(0, ax.get_ylim()[1] * 1.15)
plt.tight_layout()
plt.show()

The above bar graph helps us identify whether the dataset is balanced or unbalanced.

In [ ]:
# Correlation heatmap

plt.figure(figsize=(18, 14))
correlation = df.corr()
sns.heatmap(correlation, cmap='coolwarm', annot=True, fmt='.2f', linewidths=0.5, annot_kws={'size': 7})

plt.title('Correlation Heatmap of Phishing Website Features')
plt.tight_layout()
plt.show()

This helps us identify which features have a stronger relationship with the target and whether some features are highly correlated with one another.

In [ ]:
# Feature distribution

selected_features = [
    'UsingIP',
    'LongURL',
    'PrefixSuffix-',
    'HTTPS',
]

fig, axes = plt.subplots(2, 2, figsize=(12, 8))

for feature, ax in zip(selected_features, axes.flatten()):
    sns.countplot(data=df, x=feature, hue=feature, palette='Set2', legend=False, ax=ax)
    ax.set_title(f'Distribution of {feature}')
    ax.set_xlabel(feature)
    ax.set_ylabel('Count')

plt.tight_layout()
plt.show()

Displaying the distribution of important URL features with the count of each type of URL.
- `UsingIP`: Does the URL use an IP address instead of a domain name?
- `LongURL`: Is the URL unusally long?
- `PrefixSuffix-`: Does the domain contain a hyphen?
- `HTTPS`: Does the website use HTTPS?

## 4. Splitting the data:

The data has to be split into training and testing datasets, in a 80-20 split.

In [ ]:
# Splitting the dataset into dependant and independant fetature

X = df.drop(["class"], axis=1)
y = df["class"]

In [ ]:
# Splitting the dataset into train and test sets: 80-20 split

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [ ]:
# Checking the 80-20 split
 
print("Training feature set:", X_train.shape)
print("Training target set:", y_train.shape)

print("Testing feature set:", X_test.shape)
print("Testing target set:", y_test.shape)

In [ ]:
# Checking class distribution after splitting

print("Training Set Class Distribution:")
print(y_train.value_counts(normalize=True) * 100)

print("\nTesting Set Class Distribution:")
print(y_test.value_counts(normalize=True) * 100)

## 5. Model Building and Training:

We'll train multiple supervised machine learning algorithms and compare their performance using 5-fold Stratified Cross-Validation. The supervised machine learning models considered to train the dataset in this notebook are:
1. Logistic Regression
2. Decision Tree
3. Random Forest
4. Support Vector Machine (SVM)
5. XGBoost

The metrics considered to evaluate the model performance are Accuracy & F1 score.

In [ ]:
# Creating result holders

ML_Model = []
accuracy = []
f1_score = []
recall = []
precision = []

cv_accuracy = []
cv_f1_score = []
cv_recall = []
cv_precision = []

In [ ]:
# Function to store model results

def storeResults(model, acc, f1, rec, prec, cv_acc, cv_f1, cv_rec, cv_prec):
    ML_Model.append(model)
    
    accuracy.append(round(acc, 3))
    f1_score.append(round(f1, 3))
    recall.append(round(rec, 3))
    precision.append(round(prec, 3))

    cv_accuracy.append(round(cv_acc, 3))
    cv_f1_score.append(round(cv_f1, 3))
    cv_recall.append(round(cv_rec, 3))
    cv_precision.append(round(cv_prec, 3))

In [ ]:
# Configuring 5-fold cross-validation

from sklearn.model_selection import StratifiedKFold, cross_validate

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

scoring = {
    'accuracy': 'accuracy',
    'precision': 'precision',
    'recall': 'recall',
    'f1': 'f1'
}

### 5.1 Logistic Regression Classifier

Logistic Regression is a supervised machine learning algorithm used for classification problems. Despite its name, it is not a regression algorithm in the usual sense. It estimates the probability of a sample belonging to a particular class and uses a decision threshold to classify it.

We'll use it as our baseline model to compare against more advanced algorithms.

In [ ]:
# Importing the LogisticRegression library
from sklearn.linear_model import LogisticRegression

# Instantiating the model
log = LogisticRegression()

In [ ]:
# Performing 5 - fold cross validation

cv_log = cross_validate(
    log,
    X_train,
    y_train,
    cv=cv,
    scoring=scoring,
    n_jobs=-1
)

print("Logistic Regression - 5 Fold Cross-Validation\n")

print(f"Accuracy:  {cv_log['test_accuracy'].mean():.3f}")
print(f"Precision: {cv_log['test_precision'].mean():.3f}")
print(f"Recall:    {cv_log['test_recall'].mean():.3f}")
print(f"F1 Score:  {cv_log['test_f1'].mean():.3f}")

In [ ]:
# Training the model
log.fit(X_train,y_train)

# Generating predictions
y_train_log = log.predict(X_train)
y_test_log = log.predict(X_test)

In [ ]:
# Calculating training and testing metrics

acc_train_log = metrics.accuracy_score(y_train, y_train_log)
acc_test_log = metrics.accuracy_score(y_test, y_test_log)

f1_train_log = metrics.f1_score(y_train, y_train_log, pos_label=1)
f1_test_log = metrics.f1_score(y_test, y_test_log, pos_label=1)

recall_train_log = metrics.recall_score(y_train, y_train_log, pos_label=1)
recall_test_log = metrics.recall_score(y_test, y_test_log, pos_label=1)

precision_train_log = metrics.precision_score(y_train, y_train_log, pos_label=1)
precision_test_log = metrics.precision_score(y_test, y_test_log, pos_label=1)

print("Logistic Regression Performance\n")

print(f"Training Accuracy:  {acc_train_log:.3f}")
print(f"Testing Accuracy:   {acc_test_log:.3f}")

print(f"Training F1 Score:  {f1_train_log:.3f}")
print(f"Testing F1 Score:   {f1_test_log:.3f}")

print(f"Training Recall:    {recall_train_log:.3f}")
print(f"Testing Recall:     {recall_test_log:.3f}")

print(f"Training Precision: {precision_train_log:.3f}")
print(f"Testing Precision:  {precision_test_log:.3f}")

In [ ]:
# Computing the classification report of the model

print(metrics.classification_report(
    y_test,
    y_test_log,
    labels=[0, 1],
    target_names=['Legitimate', 'Phishing']
))

In [ ]:
# Storing the results

storeResults(
    'Logistic Regression',
    acc_test_log,
    f1_test_log,
    recall_test_log,
    precision_test_log,
    cv_log['test_accuracy'].mean(),
    cv_log['test_f1'].mean(),
    cv_log['test_recall'].mean(),
    cv_log['test_precision'].mean()
)

### 5.2 Decision Tree Classifier

A Decision Tree is a supervised machine learning algorithm used for classification and regression tasks. It makes predictions by splitting the dataset into smaller subsets based on feature values.

Each internal node represents a decision based on a feature, each branch represents an outcome, and each leaf node represents a predicted class.

In [ ]:
# Importing the DecisionTreeClassifier library
from sklearn.tree import DecisionTreeClassifier

# Initializing the model
tree = DecisionTreeClassifier(random_state=42)

In [ ]:
# Performing 5 - fold cross validation

cv_tree = cross_validate(
    tree,
    X_train,
    y_train,
    cv=cv,
    scoring=scoring,
    n_jobs=-1
)

print("Decision Tree - 5 Fold Cross-Validation\n")

print(f"Accuracy:  {cv_tree['test_accuracy'].mean():.3f}")
print(f"Precision: {cv_tree['test_precision'].mean():.3f}")
print(f"Recall:    {cv_tree['test_recall'].mean():.3f}")
print(f"F1 Score:  {cv_tree['test_f1'].mean():.3f}")

In [ ]:
# Training the model
tree.fit(X_train, y_train)

# Generating predictions
y_train_tree = tree.predict(X_train)
y_test_tree = tree.predict(X_test)

In [ ]:
# Calculating training and testing metrics

acc_train_tree = metrics.accuracy_score(y_train, y_train_tree)
acc_test_tree = metrics.accuracy_score(y_test, y_test_tree)

f1_train_tree = metrics.f1_score(y_train, y_train_tree, pos_label=1)
f1_test_tree = metrics.f1_score(y_test, y_test_tree, pos_label=1)

recall_train_tree = metrics.recall_score(y_train, y_train_tree, pos_label=1)
recall_test_tree = metrics.recall_score(y_test, y_test_tree, pos_label=1)

precision_train_tree = metrics.precision_score(y_train, y_train_tree, pos_label=1)
precision_test_tree = metrics.precision_score(y_test, y_test_tree, pos_label=1)

print("Decision Tree Performance\n")

print(f"Training Accuracy:  {acc_train_tree:.3f}")
print(f"Testing Accuracy:   {acc_test_tree:.3f}")

print(f"Training F1 Score:  {f1_train_tree:.3f}")
print(f"Testing F1 Score:   {f1_test_tree:.3f}")

print(f"Training Recall:    {recall_train_tree:.3f}")
print(f"Testing Recall:     {recall_test_tree:.3f}")

print(f"Training Precision: {precision_train_tree:.3f}")
print(f"Testing Precision:  {precision_test_tree:.3f}")

In [ ]:
# Computing the classification report of the model

print(metrics.classification_report(
    y_test,
    y_test_tree,
    labels=[0, 1],
    target_names=['Legitimate', 'Phishing']
))

In [ ]:
# Storing the results

storeResults(
    'Decision Tree',
    acc_test_tree,
    f1_test_tree,
    recall_test_tree,
    precision_test_tree,
    cv_tree['test_accuracy'].mean(),
    cv_tree['test_f1'].mean(),
    cv_tree['test_recall'].mean(),
    cv_tree['test_precision'].mean()
)

### 5.3 Random Forest Classifier

Random Forest is an ensemble machine learning algorithm that combines multiple Decision Trees to make predictions. Instead of relying on a single Decision Tree, it builds several trees using different samples of the training data and combines their predictions through majority voting.

In [ ]:
# Importing the RandomForestClassifier library
from sklearn.ensemble import RandomForestClassifier

# Initializing the model
forest = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

In [ ]:
# Performing 5 - fold cross validation

cv_forest = cross_validate(
    forest,
    X_train,
    y_train,
    cv=cv,
    scoring=scoring,
    n_jobs=-1
)

print("Random Forest - 5 Fold Cross-Validation\n")

print(f"Accuracy:  {cv_forest['test_accuracy'].mean():.3f}")
print(f"Precision: {cv_forest['test_precision'].mean():.3f}")
print(f"Recall:    {cv_forest['test_recall'].mean():.3f}")
print(f"F1 Score:  {cv_forest['test_f1'].mean():.3f}")

In [ ]:
# Training the model
forest.fit(X_train, y_train)

# Generating predictions
y_train_forest = forest.predict(X_train)
y_test_forest = forest.predict(X_test)

In [ ]:
# Calculating training and testing metrics

acc_train_forest = metrics.accuracy_score(y_train, y_train_forest)
acc_test_forest = metrics.accuracy_score(y_test, y_test_forest)

f1_train_forest = metrics.f1_score(y_train, y_train_forest, pos_label=1)
f1_test_forest = metrics.f1_score(y_test, y_test_forest, pos_label=1)

recall_train_forest = metrics.recall_score(y_train, y_train_forest, pos_label=1)
recall_test_forest = metrics.recall_score(y_test, y_test_forest, pos_label=1)

precision_train_forest = metrics.precision_score(y_train, y_train_forest, pos_label=1)
precision_test_forest = metrics.precision_score(y_test, y_test_forest, pos_label=1)

print("Random Forest Performance\n")

print(f"Training Accuracy:  {acc_train_forest:.3f}")
print(f"Testing Accuracy:   {acc_test_forest:.3f}")

print(f"Training F1 Score:  {f1_train_forest:.3f}")
print(f"Testing F1 Score:   {f1_test_forest:.3f}")

print(f"Training Recall:    {recall_train_forest:.3f}")
print(f"Testing Recall:     {recall_test_forest:.3f}")

print(f"Training Precision: {precision_train_forest:.3f}")
print(f"Testing Precision:  {precision_test_forest:.3f}")

In [ ]:
# Computing the classification report of the model

print(metrics.classification_report(
    y_test,
    y_test_forest,
    labels=[0, 1],
    target_names=['Legitimate', 'Phishing']
))

In [ ]:
# Storing the results

storeResults(
    'Random Forest',
    acc_test_forest,
    f1_test_forest,
    recall_test_forest,
    precision_test_forest,
    cv_forest['test_accuracy'].mean(),
    cv_forest['test_f1'].mean(),
    cv_forest['test_recall'].mean(),
    cv_forest['test_precision'].mean()
)

### 5.4 Support Vector Machine (SVM) Classifier

SVM is a powerful classification algorithm that finds an optimal decision boundary between classes. We'll use an RBF kernel, which can capture non-linear relationships between features.

In [ ]:
# Importing the SVM library
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

svm = Pipeline([
    ('scaler', StandardScaler()),
    ('classifier', SVC(kernel='rbf')),
    
])

In [ ]:
# Performing 5 - fold cross validation

cv_svm = cross_validate(
    svm,
    X_train,
    y_train,
    cv=cv,
    scoring=scoring,
    n_jobs=-1
)

print("SVM - 5-Fold Cross-Validation\n")
print("Accuracy :", f"{cv_svm['test_accuracy'].mean():.3f}")
print("Precision:", f"{cv_svm['test_precision'].mean():.3f}")
print("Recall   :", f"{cv_svm['test_recall'].mean():.3f}")
print("F1 Score :", f"{cv_svm['test_f1'].mean():.3f}")

In [ ]:
# Training the model
svm.fit(X_train, y_train)

# Generating predictions
y_train_svm = svm.predict(X_train)
y_test_svm = svm.predict(X_test)

In [ ]:
# Calculating training and testing metrics

acc_train_svm = metrics.accuracy_score(y_train, y_train_svm)
acc_test_svm = metrics.accuracy_score(y_test, y_test_svm)

f1_train_svm = metrics.f1_score(y_train, y_train_svm, pos_label=1)
f1_test_svm = metrics.f1_score(y_test, y_test_svm, pos_label=1)

recall_train_svm = metrics.recall_score(y_train, y_train_svm, pos_label=1)
recall_test_svm = metrics.recall_score(y_test, y_test_svm, pos_label=1)

precision_train_svm = metrics.precision_score(y_train, y_train_svm, pos_label=1)
precision_test_svm = metrics.precision_score(y_test, y_test_svm, pos_label=1)

print("Support Vector Machine Performance\n")

print("Training Accuracy :", f"{acc_train_svm:.3f}")
print("Testing Accuracy  :", f"{acc_test_svm:.3f}")

print("Training F1 Score :", f"{f1_train_svm:.3f}")
print("Testing F1 Score  :", f"{f1_test_svm:.3f}")

print("Training Recall   :", f"{recall_train_svm:.3f}")
print("Testing Recall    :", f"{recall_test_svm:.3f}")

print("Training Precision:", f"{precision_train_svm:.3f}")
print("Testing Precision :", f"{precision_test_svm:.3f}")

In [ ]:
# Computing the classification report of the model

print(metrics.classification_report(
    y_test,
    y_test_svm,
    labels=[0, 1],
    target_names=['Legitimate', 'Phishing']
))

In [ ]:
# Storing the results

storeResults(
    'SVM',
    acc_test_svm,
    f1_test_svm,
    recall_test_svm,
    precision_test_svm,
    cv_svm['test_accuracy'].mean(),
    cv_svm['test_f1'].mean(),
    cv_svm['test_recall'].mean(),
    cv_svm['test_precision'].mean()
)

### 5.5 XGBoost Classifier

XGBoost (Extreme Gradient Boosting) is an ensemble learning algorithm that builds multiple decision trees sequentially. Each new tree attempts to correct the errors made by the previous trees.

In [ ]:
# Importing the XGBClassifier Library
from xgboost import XGBClassifier

# Initializing the model
xgb = XGBClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=6,
    random_state=42,
    eval_metric='logloss',
    n_jobs=-1
)

In [ ]:
# Performing 5 - fold cross validation

cv_xgb = cross_validate(
    xgb,
    X_train,
    y_train,
    cv=cv,
    scoring=scoring,
    n_jobs=-1
)

print("XGBoost 5-Fold Cross-Validation\n")
print("Accuracy :", f"{cv_xgb['test_accuracy'].mean():.3f}")
print("Precision:", f"{cv_xgb['test_precision'].mean():.3f}")
print("Recall   :", f"{cv_xgb['test_recall'].mean():.3f}")
print("F1 Score :", f"{cv_xgb['test_f1'].mean():.3f}")

In [ ]:
# Training the model
xgb.fit(X_train, y_train)

# Generating predictions
y_train_xgb = xgb.predict(X_train)
y_test_xgb = xgb.predict(X_test)

In [ ]:
# Calculating training and testing metrics

acc_train_xgb = metrics.accuracy_score(y_train, y_train_xgb)
acc_test_xgb = metrics.accuracy_score(y_test, y_test_xgb)

f1_train_xgb = metrics.f1_score(y_train, y_train_xgb, pos_label=1)
f1_test_xgb = metrics.f1_score(y_test, y_test_xgb, pos_label=1)

recall_train_xgb = metrics.recall_score(y_train, y_train_xgb, pos_label=1)
recall_test_xgb = metrics.recall_score(y_test, y_test_xgb, pos_label=1)

precision_train_xgb = metrics.precision_score(y_train, y_train_xgb, pos_label=1)
precision_test_xgb = metrics.precision_score(y_test, y_test_xgb, pos_label=1)

print("XGBoost Performance\n")

print("Training Accuracy :", f"{acc_train_xgb:.3f}")
print("Testing Accuracy  :", f"{acc_test_xgb:.3f}")

print("Training F1 Score :", f"{f1_train_xgb:.3f}")
print("Testing F1 Score  :", f"{f1_test_xgb:.3f}")

print("Training Recall   :", f"{recall_train_xgb:.3f}")
print("Testing Recall    :", f"{recall_test_xgb:.3f}")

print("Training Precision:", f"{precision_train_xgb:.3f}")
print("Testing Precision :", f"{precision_test_xgb:.3f}")

In [ ]:
# Computing the classification report of the model

print(metrics.classification_report(
    y_test,
    y_test_xgb,
    labels=[0, 1],
    target_names=['Legitimate', 'Phishing'],
    digits=3
))

In [ ]:
# Storing the results

storeResults(
    'XGBoost',
    acc_test_xgb,
    f1_test_xgb,
    recall_test_xgb,
    precision_test_xgb,
    cv_xgb['test_accuracy'].mean(),
    cv_xgb['test_f1'].mean(),
    cv_xgb['test_recall'].mean(),
    cv_xgb['test_precision'].mean()
)

## 6. Comparison of Models

To compare the models performance, a dataframe is created. The columns of this dataframe are the lists created to store the results of the model.

In [ ]:
# Creating a comparison DataFrame

result = pd.DataFrame({
    'ML Model': ML_Model,

    'Test Accuracy': accuracy,
    'Test F1 Score': f1_score,
    'Test Recall': recall,
    'Test Precision': precision,

    'CV Accuracy': cv_accuracy,
    'CV F1 Score': cv_f1_score,
    'CV Recall': cv_recall,
    'CV Precision': cv_precision
})

# Displaying the result
result

In [ ]:
# Sorting the models
sorted_result = result.sort_values(
    by=['CV F1 Score', 'CV Recall', 'CV Accuracy'],
    ascending=False
).reset_index(drop=True)

# Displaying the result
sorted_result

## 7. Storing the Best Model

In [ ]:
models = {
    'Logistic Regression': log,
    'Decision Tree': tree,
    'Random Forest': forest,
    'SVM': svm,
    'XGBoost': xgb
}

best_model_name = ML_Model[cv_f1_score.index(max(cv_f1_score))]
best_model = models[best_model_name]

print("Best Model:", best_model_name)
print("Best CV F1 Score:", f"{max(cv_f1_score):.3f}")

In [ ]:
# Training based on the best model we got
best_model.fit(X_train, y_train)

In [ ]:
# Saving the best model using pickle

import pickle
import os

os.makedirs('pickle', exist_ok=True)

with open('pickle/best_model.pkl', 'wb') as file: pickle.dump(best_model, file)

print("Best model saved successfully!")

## 8. Feature Importance Analysis

In [ ]:
# Using Permutation Feature Importance to find the best features
 
from sklearn.inspection import permutation_importance

importance = permutation_importance(
    best_model,
    X_train,
    y_train,
    n_repeats=10,
    random_state=42,
    scoring='f1',
    n_jobs=-1
)

In [ ]:
feature_importance = pd.DataFrame({
    'Feature': X_test.columns,
    'Importance': importance.importances_mean
})

feature_importance = feature_importance.sort_values(
    by='Importance',
    ascending=False
).reset_index(drop=True)

feature_importance.head(15)

In [ ]:
# Visualizing the Top 15 features

plt.figure(figsize=(10, 7))

top_features = feature_importance.head(15).sort_values(
    by='Importance',
    ascending=True
)

plt.barh(
    top_features['Feature'],
    top_features['Importance']
)

plt.title('Top 15 Features - Permutation Importance')
plt.xlabel('Mean Decrease in F1 Score')
plt.ylabel('Features')
plt.tight_layout()
plt.show()

**What this means:** A feature with a higher permutation importance has a greater impact on the model's F1 score. If changing or shuffling a feature causes a substantial drop in F1 Score, that feature is useful to the model.

## 9. Conclusion

1. Developing this notebook was helpful in making me gain practical knowledge of how different website features affect machine learning models in distinguishing phishing websites from legitimate ones. I also learned how to train, evaluate, and compare multiple models using accuracy, precision, recall, F1-score, and 5-fold cross-validation.
2. The feature importance analysis helped identify the website characteristics that contribute most to the selected model's predictions. Features such as HTTPS, AnchorURL, WebsiteTraffic, showed relatively greater importance in identifying phishing websites.
3. By comparing Logistic Regression, Decision Tree, Random Forest, SVM, and XGBoost, I was able to identify the best-performing model based on its cross-validation F1-score and evaluate its performance on unseen test data.
4. The 'Random Forest' model achieved a test accuracy of 97.3%, with an F1-score of 0.964, demonstrating its effectiveness in distinguishing phishing websites from legitimate websites in the given dataset.

In [ ]:
# Run this cell at the END of your completed notebook.
# It exports the five already-trained models and their cross-validation F1 scores.

import os
import pickle

models = {
    'Logistic Regression': log,
    'Decision Tree': tree,
    'Random Forest': forest,
    'SVM': svm,
    'XGBoost': xgb
}

cv_f1_scores = {
    'Logistic Regression': float(cv_log['test_f1'].mean()),
    'Decision Tree': float(cv_tree['test_f1'].mean()),
    'Random Forest': float(cv_forest['test_f1'].mean()),
    'SVM': float(cv_svm['test_f1'].mean()),
    'XGBoost': float(cv_xgb['test_f1'].mean())
}

best_model_name = max(cv_f1_scores, key=cv_f1_scores.get)

bundle = {
    'models': models,
    'best_model_name': best_model_name,
    'cv_f1_scores': cv_f1_scores,
    'feature_names': list(X_train.columns)
}

os.makedirs('model', exist_ok=True)

with open('model/phishguard_models.pkl', 'wb') as f:
    pickle.dump(bundle, f)

print("Exported model bundle successfully.")
print("Best model:", best_model_name)
print("CV F1 scores:", cv_f1_scores)
